## Project Overview

This project analyses a synthetic healthcare clinic dataset using Python, Excel and Tableau Public.

The objective is to understand patterns in patient activity, appointments, clinical services and recorded payment values, and to develop an interactive business intelligence dashboard for operational analysis.

### Business Question

> How are patient activity, clinical services and healthcare payment activity distributed over time and across clinics, doctors and specialties?

### Tools

- Python — data inspection, cleaning and validation
- Excel — analytical summaries and KPI validation
- Tableau Public — interactive business intelligence dashboard

### Dataset

The dataset contains synthetic healthcare clinic records covering:

- Patients
- Appointments
- Doctors
- Clinics
- Payments
- Treatments
- Prescriptions
- Staff
- Patient insurance

The data is synthetic and should not be interpreted as representative of real-world healthcare activity.

### Dashboard Focus

The final business intelligence dashboard will examine:

- Appointment activity and status
- Patient utilisation
- Clinic and doctor activity
- Treatment patterns
- Prescription activity
- Insurance coverage
- Payment methods
- Treatment costs
- Trends over time

The dashboard is designed to demonstrate healthcare analytics and business intelligence skills using a synthetic dataset.

## Data Sources

In [1]:
import pandas as pd
from pathlib import Path

# Project data directory
data_path = Path("../data/raw")

# Load source tables
appointments = pd.read_csv(data_path / "appointments.csv")
clinics = pd.read_csv(data_path / "clinics.csv")
doctors = pd.read_csv(data_path / "doctors.csv")
patients = pd.read_csv(data_path / "patients.csv")
payments = pd.read_csv(data_path / "payments.csv")
prescriptions = pd.read_csv(data_path / "prescriptions.csv")
staff = pd.read_csv(data_path / "staff.csv")

treatments = pd.read_csv(data_path / "treatments.csv")

insurance = pd.read_csv(
    data_path / "patient_insurance.txt",
    sep="|",
    header=None,
    names=[
        "patient_id",
        "insurance_status",
        "insurance_provider",
        "coverage_level"
    ]
)

print("Data loaded successfully.")

Data loaded successfully.


The project uses nine source files from the Kaggle Healthcare Clinic Dataset.

| Source | Records | Role |
|---|---:|---|
| `patients.csv` | 20,000 | Patient information |
| `appointments.csv` | 100,000 | Appointment activity |
| `doctors.csv` | 500 | Doctor information |
| `clinics.csv` | 50 | Clinic information |
| `payments.csv` | 100,000 | Payment transactions |
| `treatments.csv` | 100,000 | Treatment activity and costs |
| `prescriptions.csv` | 100,000 | Prescription records |
| `staff.csv` | 1,000 | Clinic staff information |
| `patient_insurance.txt` | 20,000 | Patient insurance information |

The dataset is synthetic. The records are therefore used for analytical and dashboard-development practice rather than for drawing conclusions about real-world healthcare populations or clinical outcomes.

## Data Quality Checks

Before cleaning or transforming the data, the source tables are checked for:

- Record counts
- Missing values
- Duplicate primary identifiers
- Valid relationships between tables
- Appropriate data types for dates and numeric measures

In [2]:
# Summarise the size and missing values of each source table

tables = {
    "Patients": patients,
    "Appointments": appointments,
    "Doctors": doctors,
    "Clinics": clinics,
    "Payments": payments,
    "Treatments": treatments,
    "Prescriptions": prescriptions,
    "Staff": staff,
    "Insurance": insurance
}

quality_summary = pd.DataFrame({
    "Rows": {name: len(df) for name, df in tables.items()},
    "Columns": {name: len(df.columns) for name, df in tables.items()},
    "Missing Values": {name: int(df.isna().sum().sum()) for name, df in tables.items()},
    "Duplicate Rows": {name: int(df.duplicated().sum()) for name, df in tables.items()}
})

quality_summary

,Rows,Columns,Missing Values,Duplicate Rows
Patients,20000,10,0,0
Appointments,100000,7,0,0
Doctors,500,6,0,0
Clinics,50,8,0,0
Payments,100000,6,0,0
Treatments,100000,5,0,0
Prescriptions,100000,4,0,0
Staff,1000,6,0,0
Insurance,20000,4,0,0


In [3]:
# Check uniqueness of the main identifier in each table

primary_keys = {
    "Patients": ("patient_id", patients),
    "Appointments": ("appointment_id", appointments),
    "Doctors": ("doctor_id", doctors),
    "Clinics": ("clinic_id", clinics),
    "Payments": ("payment_id", payments),
    "Treatments": ("treatment_id", treatments),
    "Prescriptions": ("prescription_id", prescriptions),
    "Staff": ("staff_id", staff),
    "Insurance": ("patient_id", insurance)
}

key_summary = pd.DataFrame([
    {
        "Table": name,
        "Primary Key": column,
        "Rows": len(df),
        "Unique Values": df[column].nunique(),
        "Duplicate Keys": int(df[column].duplicated().sum())
    }
    for name, (column, df) in primary_keys.items()
])

key_summary

,Table,Primary Key,Rows,Unique Values,Duplicate Keys
0,Patients,patient_id,20000,20000,0
1,Appointments,appointment_id,100000,100000,0
2,Doctors,doctor_id,500,500,0
3,Clinics,clinic_id,50,50,0
4,Payments,payment_id,100000,100000,0
5,Treatments,treatment_id,100000,100000,0
6,Prescriptions,prescription_id,100000,100000,0
7,Staff,staff_id,1000,1000,0
8,Insurance,patient_id,20000,20000,0


## Data Preparation

The source files contain complete records and unique primary identifiers. The next stage is to inspect date fields, categorical variables and numeric measures before creating the cleaned analytical tables.

The raw files are preserved unchanged in `data/raw/`. Any transformations will be written to `data/cleaned/`.

In [4]:
# Inspect date columns and their current data types

date_columns = {
    "Patients": ["date_of_birth", "registration_date"],
    "Appointments": ["appointment_date"],
    "Payments": ["payment_date"]
}

for table_name, columns in date_columns.items():
    df = tables[table_name]
    
    print(f"\n{'='*60}")
    print(table_name.upper())
    print(f"{'='*60}")
    
    for column in columns:
        print(f"\n{column}")
        print("Data type:", df[column].dtype)
        print("Example values:", df[column].head(5).tolist())


PATIENTS

date_of_birth
Data type: object
Example values: ['11/02/2011', '12/12/2002', '03/30/1939', '04/09/1984', '06/24/2020']

registration_date
Data type: object
Example values: ['10/25/2024', '05/14/2025', '12/13/2024', '07/20/2024', '06/12/2025']

APPOINTMENTS

appointment_date
Data type: object
Example values: ['07/31/2025', '03/29/2025', '07/10/2025', '06/13/2025', '12/03/2025']

PAYMENTS

payment_date
Data type: object
Example values: ['06/10/2025', '10/22/2025', '03/26/2025', '12/02/2025', '12/04/2025']


In [5]:
# Inspect the unique values in important categorical fields

categorical_columns = {
    "Appointments": ["status", "reason_for_visit"],
    "Patients": ["gender", "blood_group"],
    "Doctors": ["specialty"],
    "Clinics": ["clinic_type"],
    "Payments": ["payment_method", "payment_status"],
    "Treatments": ["treatment_type", "treatment_status"],
    "Insurance": ["insurance_status", "insurance_provider", "coverage_level"],
    "Staff": ["role"]
}

for table_name, columns in categorical_columns.items():
    df = tables[table_name]
    
    print(f"\n{'='*60}")
    print(table_name.upper())
    print(f"{'='*60}")
    
    for column in columns:
        print(f"\n{column}:")
        print(df[column].value_counts().to_string())


APPOINTMENTS

status:
status
Pending      33439
Completed    33409
Cancelled    33152

reason_for_visit:
reason_for_visit
Skin Allergy       16868
General Checkup    16716
Back Pain          16650
Fever              16619
Follow-up Visit    16596
Heart Pain         16551

PATIENTS

gender:
gender
Female    10122
Male       9878

blood_group:
blood_group
O+     5064
A+     5013
AB+    4984
B+     4939

DOCTORS

specialty:
specialty
Dermatology    172
Pediatrics     168
Cardiology     160

CLINICS

clinic_type:
clinic_type
Public     30
Private    20

PAYMENTS

payment_method:
payment_method
Card         33489
Cash         33482
Insurance    33029

payment_status:
payment_status
Paid    100000

TREATMENTS

treatment_type:
treatment_type
Surgery    33420
Checkup    33312
Therapy    33268

treatment_status:
treatment_status
Pending      33595
Completed    33353
Ongoing      33052

INSURANCE

insurance_status:
insurance_status
Not Insured    10057
Insured         9943

insurance_provider:


## Date Preparation

The source tables store dates as text in `MM/DD/YYYY` format. These fields are converted to pandas datetime values so that they can be used reliably for chronological analysis, monthly trends and Power BI date filtering.

The original raw files remain unchanged.

In [6]:
# Convert date fields to datetime

patients["date_of_birth"] = pd.to_datetime(
    patients["date_of_birth"],
    format="%m/%d/%Y"
)

patients["registration_date"] = pd.to_datetime(
    patients["registration_date"],
    format="%m/%d/%Y"
)

appointments["appointment_date"] = pd.to_datetime(
    appointments["appointment_date"],
    format="%m/%d/%Y"
)

payments["payment_date"] = pd.to_datetime(
    payments["payment_date"],
    format="%m/%d/%Y"
)

print("Date conversion completed.")

Date conversion completed.


In [7]:
print(patients[["date_of_birth", "registration_date"]].dtypes)
print(appointments["appointment_date"].dtype)
print(payments["payment_date"].dtype)

date_of_birth        datetime64[ns]
registration_date    datetime64[ns]
dtype: object
datetime64[ns]
datetime64[ns]


In [8]:
print("Patient registration dates:")
print(patients["registration_date"].min(), "to", patients["registration_date"].max())

print("\nAppointment dates:")
print(appointments["appointment_date"].min(), "to", appointments["appointment_date"].max())

print("\nPayment dates:")
print(payments["payment_date"].min(), "to", payments["payment_date"].max())

print("\nPatient birth dates:")
print(patients["date_of_birth"].min(), "to", patients["date_of_birth"].max())

Patient registration dates:
2024-03-22 00:00:00 to 2026-03-22 00:00:00

Appointment dates:
2025-03-22 00:00:00 to 2026-03-22 00:00:00

Payment dates:
2025-03-22 00:00:00 to 2026-03-22 00:00:00

Patient birth dates:
1910-03-28 00:00:00 to 2026-03-22 00:00:00


## Financial Data Validation

Treatment cost and payment amount are the primary financial measures in the dataset.

Before creating financial KPIs, their ranges and distributions are checked, along with the relationship between appointment, treatment and payment dates.

In [9]:
# Inspect financial measures

print("Treatment cost:")
print(treatments["treatment_cost"].describe())

print("\nPayment amount:")
print(payments["payment_amount"].describe())

print("\nPayment methods:")
print(payments["payment_method"].value_counts())

print("\nPayment status:")
print(payments["payment_status"].value_counts())

Treatment cost:
count    100000.000000
mean        801.381376
std         403.741509
min         100.000000
25%         452.367500
50%         800.600000
75%        1150.772500
max        1499.990000
Name: treatment_cost, dtype: float64

Payment amount:
count    100000.000000
mean        801.381376
std         403.741509
min         100.000000
25%         452.367500
50%         800.600000
75%        1150.772500
max        1499.990000
Name: payment_amount, dtype: float64

Payment methods:
payment_method
Card         33489
Cash         33482
Insurance    33029
Name: count, dtype: int64

Payment status:
payment_status
Paid    100000
Name: count, dtype: int64


In [10]:
# Compare appointment, treatment and payment dates

date_comparison = appointments[["appointment_id", "appointment_date"]].merge(
    payments[["appointment_id", "payment_date"]],
    on="appointment_id"
).merge(
    treatments[["appointment_id"]],
    on="appointment_id"
)

date_comparison["payment_after_appointment"] = (
    date_comparison["payment_date"] >= date_comparison["appointment_date"]
)

print("Payments on or after appointment date:",
      date_comparison["payment_after_appointment"].sum())

print("Payments before appointment date:",
      (~date_comparison["payment_after_appointment"]).sum())

Payments on or after appointment date: 49967
Payments before appointment date: 50033


## Table Structure

The following inspection records the columns available in each source table. This provides the basis for defining relationships, cleaning rules and the Power BI data model.

In [11]:
for table_name, df in tables.items():
    print(f"\n{'='*60}")
    print(table_name.upper())
    print(f"{'='*60}")
    print(df.columns.tolist())


PATIENTS
['patient_id', 'full_name', 'gender', 'date_of_birth', 'phone', 'email', 'address', 'registration_date', 'blood_group', 'emergency_contact']

APPOINTMENTS
['appointment_id', 'patient_id', 'doctor_id', 'clinic_id', 'appointment_date', 'status', 'reason_for_visit']

DOCTORS
['doctor_id', 'full_name', 'specialty', 'phone', 'email', 'clinic_id']

CLINICS
['clinic_id', 'clinic_name', 'address', 'city', 'state', 'phone', 'opening_hours', 'clinic_type']

PAYMENTS
['payment_id', 'appointment_id', 'payment_date', 'payment_amount', 'payment_method', 'payment_status']

TREATMENTS
['treatment_id', 'appointment_id', 'treatment_type', 'treatment_cost', 'treatment_status']

PRESCRIPTIONS
['prescription_id', 'appointment_id', 'medicine_name', 'instructions']

STAFF
['staff_id', 'full_name', 'role', 'clinic_id', 'phone', 'email']

INSURANCE
['patient_id', 'insurance_status', 'insurance_provider', 'coverage_level']


In [12]:
print("Appointments:", len(appointments))
print("Unique appointment IDs:", appointments["appointment_id"].nunique())

print("\nAppointments per patient:")
print(appointments["patient_id"].value_counts().describe())

print("\nAppointments per doctor:")
print(appointments["doctor_id"].value_counts().describe())

print("\nAppointments per clinic:")
print(appointments["clinic_id"].value_counts().describe())

Appointments: 100000
Unique appointment IDs: 100000

Appointments per patient:
count    19861.000000
mean         5.034993
std          2.222320
min          1.000000
25%          3.000000
50%          5.000000
75%          6.000000
max         16.000000
Name: count, dtype: float64

Appointments per doctor:
count    500.000000
mean     200.000000
std       13.940469
min      150.000000
25%      190.000000
50%      200.000000
75%      209.000000
max      243.000000
Name: count, dtype: float64

Appointments per clinic:
count      50.00000
mean     2000.00000
std        43.30292
min      1903.00000
25%      1964.50000
50%      1999.00000
75%      2037.25000
max      2081.00000
Name: count, dtype: float64


## Data Understanding Summary

The dataset contains nine relational tables covering patients, appointments, doctors, clinics, payments, treatments, prescriptions, staff and insurance.

The source tables contain complete records with no missing values or duplicate rows, and all tested primary and foreign-key relationships are valid.

The `appointments` table provides the central activity record for the dashboard. It contains 100,000 unique appointments linked to 20,000 patients, 500 doctors and 50 clinics. Most patients have multiple appointments, with an average of approximately five appointments per patient among patients with recorded appointments.

The appointment and payment activity covers 22 March 2025 to 22 March 2026. Patient registration dates begin one year earlier, while birth dates span a very wide range and include implausible synthetic edge cases.

Several categorical variables are unusually evenly distributed. This is consistent with the dataset being synthetic and means that dashboard findings should be interpreted as examples of healthcare analytics rather than evidence about real-world healthcare activity.

Payment amounts and treatment costs have identical distributions, and all payment records are marked as paid. Payment dates occur before the appointment date for approximately half of the records, so payment timing will not be interpreted as a real-world revenue-recognition measure.

The raw source files will remain unchanged. Cleaned analytical tables will be created separately for Excel and Power BI.

## Proposed Power BI Data Model

The dashboard will retain the source tables as separate entities rather than flattening the dataset into one large table.

The appointment table will act as the central activity table, with patient, doctor and clinic information providing descriptive dimensions. Payments, treatments and prescriptions will remain linked to appointments. Insurance will be linked to patients, while staff will be linked to clinics.

This structure preserves the relational nature of the dataset and reduces unnecessary duplication when building the Power BI model.

## Cleaned Analytical Tables

The raw source files are preserved unchanged in `data/raw/`.

For analysis and dashboard development, cleaned copies of the source tables are created in `data/cleaned/`. Dates are stored as datetime values, while the relational structure of the source data is retained.

No records are intentionally removed at this stage because the source tables contain complete records and the tested relationships are valid.

In [13]:
from pathlib import Path

cleaned_path = Path("../data/cleaned")
cleaned_path.mkdir(parents=True, exist_ok=True)

# Save cleaned analytical tables
patients.to_csv(cleaned_path / "patients_clean.csv", index=False)
appointments.to_csv(cleaned_path / "appointments_clean.csv", index=False)
doctors.to_csv(cleaned_path / "doctors_clean.csv", index=False)
clinics.to_csv(cleaned_path / "clinics_clean.csv", index=False)
payments.to_csv(cleaned_path / "payments_clean.csv", index=False)
treatments.to_csv(cleaned_path / "treatments_clean.csv", index=False)
prescriptions.to_csv(cleaned_path / "prescriptions_clean.csv", index=False)
staff.to_csv(cleaned_path / "staff_clean.csv", index=False)
insurance.to_csv(cleaned_path / "patient_insurance_clean.csv", index=False)

print("Cleaned analytical tables saved successfully.")

Cleaned analytical tables saved successfully.


In [14]:
for file in sorted(cleaned_path.iterdir()):
    print(file.name)

appointments_clean.csv
blood_group_summary.csv
clinic_activity.csv
clinics_clean.csv
coverage_summary.csv
doctor_activity.csv
doctors_clean.csv
financial_summary.csv
gender_summary.csv
insurance_provider_summary.csv
insurance_summary.csv
kpi_summary.csv
monthly_appointments.csv
monthly_status.csv
patient_insurance_clean.csv
patients_clean.csv
payment_summary.csv
payments_clean.csv
prescriptions_clean.csv
specialty_activity.csv
staff_clean.csv
treatment_status_summary.csv
treatment_summary.csv
treatments_clean.csv


## Core KPI Validation

The following measures establish the baseline figures that the final Excel and Power BI dashboards should reproduce.

In [15]:
total_appointments = len(appointments)

completed_appointments = (
    appointments["status"].eq("Completed").sum()
)

cancelled_appointments = (
    appointments["status"].eq("Cancelled").sum()
)

pending_appointments = (
    appointments["status"].eq("Pending").sum()
)

unique_patients = appointments["patient_id"].nunique()

total_treatment_cost = treatments["treatment_cost"].sum()

average_treatment_cost = treatments["treatment_cost"].mean()

total_payments = payments["payment_amount"].sum()

average_payment = payments["payment_amount"].mean()

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total appointments",
        "Completed appointments",
        "Cancelled appointments",
        "Pending appointments",
        "Unique patients with appointments",
        "Total treatment cost",
        "Average treatment cost",
        "Total recorded payments",
        "Average payment"
    ],
    "Value": [
        total_appointments,
        completed_appointments,
        cancelled_appointments,
        pending_appointments,
        unique_patients,
        total_treatment_cost,
        average_treatment_cost,
        total_payments,
        average_payment
    ]
})

kpi_summary

,Metric,Value
0,Total appointments,1.000000e+05
1,Completed appointments,3.340900e+04
2,Cancelled appointments,3.315200e+04
3,Pending appointments,3.343900e+04
4,Unique patients with appointments,1.986100e+04
5,Total treatment cost,8.013814e+07
6,Average treatment cost,8.013814e+02
7,Total recorded payments,8.013814e+07
8,Average payment,8.013814e+02


In [16]:
completion_rate = completed_appointments / total_appointments * 100

cancellation_rate = cancelled_appointments / total_appointments * 100

print(f"Completion rate: {completion_rate:.2f}%")
print(f"Cancellation rate: {cancellation_rate:.2f}%")

Completion rate: 33.41%
Cancellation rate: 33.15%


In [17]:
appointments["appointment_month"] = (
    appointments["appointment_date"]
    .dt.to_period("M")
    .astype(str)
)

monthly_appointments = (
    appointments
    .groupby("appointment_month")
    .size()
    .reset_index(name="appointments")
)

monthly_appointments

,appointment_month,appointments
0,2025-03,2785
1,2025-04,8231
2,2025-05,8382
3,2025-06,8036
4,2025-07,8452
5,2025-08,8574
6,2025-09,8112
7,2025-10,8371
8,2025-11,8331
9,2025-12,8637


In [18]:
monthly_status = (
    appointments
    .groupby(["appointment_month", "status"])
    .size()
    .reset_index(name="appointments")
)

monthly_status

,appointment_month,status,appointments
0,2025-03,Cancelled,855
1,2025-03,Completed,950
2,2025-03,Pending,980
3,2025-04,Cancelled,2689
4,2025-04,Completed,2807
5,2025-04,Pending,2735
6,2025-05,Cancelled,2815
7,2025-05,Completed,2720
8,2025-05,Pending,2847
9,2025-06,Cancelled,2690


## Clinic Activity Analysis

Clinic activity is analysed using appointment volume and appointment status.

Because the dataset is synthetic and clinic volumes are relatively balanced, the analysis describes the distribution of activity rather than implying meaningful differences in real-world clinic performance.

In [19]:
clinic_activity = (
    appointments
    .groupby("clinic_id")
    .agg(
        total_appointments=("appointment_id", "count"),
        completed=("status", lambda x: (x == "Completed").sum()),
        cancelled=("status", lambda x: (x == "Cancelled").sum()),
        pending=("status", lambda x: (x == "Pending").sum())
    )
    .reset_index()
)

clinic_activity["completion_rate"] = (
    clinic_activity["completed"]
    / clinic_activity["total_appointments"]
    * 100
)

clinic_activity["cancellation_rate"] = (
    clinic_activity["cancelled"]
    / clinic_activity["total_appointments"]
    * 100
)

clinic_activity.sort_values(
    "total_appointments",
    ascending=False
).head(10)

,clinic_id,total_appointments,completed,cancelled,pending,completion_rate,cancellation_rate
16,C24,2081,667,699,715,32.051898,33.589620
13,C21,2072,707,696,669,34.121622,33.590734
43,C49,2067,685,692,690,33.139816,33.478471
18,C26,2059,679,677,703,32.977173,32.880039
26,C33,2054,664,652,738,32.327167,31.742941
2,C11,2053,703,692,658,34.242572,33.706771
11,C2,2052,659,687,706,32.115010,33.479532
40,C46,2047,689,695,663,33.659013,33.952125
5,C14,2046,707,670,669,34.555230,32.746823
37,C43,2043,694,668,681,33.969652,32.697014


In [20]:
clinic_activity = clinic_activity.merge(
    clinics[["clinic_id", "clinic_name", "city", "clinic_type"]],
    on="clinic_id",
    how="left"
)

clinic_activity = clinic_activity[
    [
        "clinic_id",
        "clinic_name",
        "city",
        "clinic_type",
        "total_appointments",
        "completed",
        "cancelled",
        "pending",
        "completion_rate",
        "cancellation_rate"
    ]
]

clinic_activity.head()

,clinic_id,clinic_name,city,clinic_type,total_appointments,completed,cancelled,pending,completion_rate,cancellation_rate
0,C1,Medical Group Smithfurt,East Mistybury,Private,2007,690,623,694,34.379671,31.041355
1,C10,HealthCare Center East Matthewshire,West Ryan,Private,1983,671,674,638,33.837620,33.988906
2,C11,HealthCare Center East Scottstad,South Christinaburgh,Public,2053,703,692,658,34.242572,33.706771
3,C12,Family Health Clinic Butlerchester,Jenniferborough,Public,1957,656,658,643,33.520695,33.622892
4,C13,HealthCare Center East Lauraport,West Stevenhaven,Private,2035,688,702,645,33.808354,34.496314


## Doctor and Specialty Activity

Doctor workload is assessed through appointment volume, while specialty-level activity provides a higher-level view of clinical demand.

The analysis uses appointment counts rather than attempting to assess clinical quality or doctor performance.

In [21]:
doctor_activity = (
    appointments
    .groupby("doctor_id")
    .agg(
        total_appointments=("appointment_id", "count"),
        completed=("status", lambda x: (x == "Completed").sum()),
        cancelled=("status", lambda x: (x == "Cancelled").sum()),
        pending=("status", lambda x: (x == "Pending").sum())
    )
    .reset_index()
)

doctor_activity["completion_rate"] = (
    doctor_activity["completed"]
    / doctor_activity["total_appointments"]
    * 100
)

doctor_activity = doctor_activity.merge(
    doctors[["doctor_id", "full_name", "specialty", "clinic_id"]],
    on="doctor_id",
    how="left"
)

doctor_activity.head()

,doctor_id,total_appointments,completed,cancelled,pending,completion_rate,full_name,specialty,clinic_id
0,D1,203,60,68,75,29.556650,Daniel Parks,Dermatology,C25
1,D10,204,73,64,67,35.784314,Amanda Duncan,Pediatrics,C39
2,D100,176,49,69,58,27.840909,Chad Mitchell,Dermatology,C42
3,D101,179,64,61,54,35.754190,Frederick Williams,Pediatrics,C48
4,D102,205,69,73,63,33.658537,Erik Joseph,Pediatrics,C5


In [22]:
specialty_activity = (
    appointments
    .merge(
        doctors[["doctor_id", "specialty"]],
        on="doctor_id",
        how="left"
    )
    .groupby("specialty")
    .agg(
        total_appointments=("appointment_id", "count"),
        completed=("status", lambda x: (x == "Completed").sum()),
        cancelled=("status", lambda x: (x == "Cancelled").sum()),
        pending=("status", lambda x: (x == "Pending").sum())
    )
    .reset_index()
)

specialty_activity["completion_rate"] = (
    specialty_activity["completed"]
    / specialty_activity["total_appointments"]
    * 100
)

specialty_activity

,specialty,total_appointments,completed,cancelled,pending,completion_rate
0,Cardiology,32023,10718,10662,10643,33.469694
1,Dermatology,34411,11490,11467,11454,33.390486
2,Pediatrics,33566,11201,11023,11342,33.370077


## Treatment Activity and Cost Analysis

Treatment records are linked to appointments and contain treatment type, treatment status and treatment cost.

The analysis examines treatment volume, status and recorded cost. Because the dataset is synthetic, treatment cost is treated as a recorded financial measure rather than evidence of real-world healthcare pricing.

In [23]:
treatment_summary = (
    treatments
    .groupby("treatment_type")
    .agg(
        treatments=("treatment_id", "count"),
        completed=("treatment_status", lambda x: (x == "Completed").sum()),
        ongoing=("treatment_status", lambda x: (x == "Ongoing").sum()),
        pending=("treatment_status", lambda x: (x == "Pending").sum()),
        total_cost=("treatment_cost", "sum"),
        average_cost=("treatment_cost", "mean")
    )
    .reset_index()
)

treatment_summary

,treatment_type,treatments,completed,ongoing,pending,total_cost,average_cost
0,Checkup,33312,10950,11098,11264,26653811.11,800.126414
1,Surgery,33420,11256,10987,11177,26782162.81,801.381293
2,Therapy,33268,11147,10967,11154,26702163.72,802.638082


In [24]:
treatment_status_summary = (
    treatments["treatment_status"]
    .value_counts()
    .rename_axis("treatment_status")
    .reset_index(name="treatments")
)

treatment_status_summary

,treatment_status,treatments
0,Pending,33595
1,Completed,33353
2,Ongoing,33052


## Payment Analysis

Payment records contain the recorded payment amount, payment method and payment status.

All payment records are marked as paid. Payment amount has the same distribution as treatment cost, so the dashboard will describe this as recorded payment value rather than making assumptions about accounting revenue recognition.

In [25]:
payment_summary = (
    payments
    .groupby("payment_method")
    .agg(
        transactions=("payment_id", "count"),
        total_payment=("payment_amount", "sum"),
        average_payment=("payment_amount", "mean")
    )
    .reset_index()
)

payment_summary

,payment_method,transactions,total_payment,average_payment
0,Card,33489,26813817.66,800.675376
1,Cash,33482,26865284.97,802.379935
2,Insurance,33029,26459035.01,801.084956


In [26]:
financial_summary = pd.DataFrame({
    "Metric": [
        "Total treatment cost",
        "Average treatment cost",
        "Total recorded payment value",
        "Average payment",
        "Number of payment transactions"
    ],
    "Value": [
        treatments["treatment_cost"].sum(),
        treatments["treatment_cost"].mean(),
        payments["payment_amount"].sum(),
        payments["payment_amount"].mean(),
        len(payments)
    ]
})

financial_summary

,Metric,Value
0,Total treatment cost,8.013814e+07
1,Average treatment cost,8.013814e+02
2,Total recorded payment value,8.013814e+07
3,Average payment,8.013814e+02
4,Number of payment transactions,1.000000e+05


## Insurance Analysis

Insurance information is linked to patients through `patient_id`.

The analysis examines insurance status, provider and coverage level. These measures describe the composition of the synthetic dataset and should not be interpreted as representative of an actual patient population.

In [27]:
insurance_summary = (
    insurance
    .groupby("insurance_status")
    .size()
    .reset_index(name="patients")
)

insurance_summary

,insurance_status,patients
0,Insured,9943
1,Not Insured,10057


In [28]:
insurance_provider_summary = (
    insurance
    .groupby("insurance_provider")
    .size()
    .reset_index(name="patients")
)

insurance_provider_summary

,insurance_provider,patients
0,Aetna,9903
1,Cigna,10097


In [29]:
coverage_summary = (
    insurance
    .groupby("coverage_level")
    .size()
    .reset_index(name="patients")
)

coverage_summary

,coverage_level,patients
0,High,6636
1,Low,6577
2,Medium,6787


In [30]:
gender_summary = (
    patients["gender"]
    .value_counts()
    .rename_axis("gender")
    .reset_index(name="patients")
)

blood_group_summary = (
    patients["blood_group"]
    .value_counts()
    .rename_axis("blood_group")
    .reset_index(name="patients")
)

gender_summary

,gender,patients
0,Female,10122
1,Male,9878


In [31]:
blood_group_summary

,blood_group,patients
0,O+,5064
1,A+,5013
2,AB+,4984
3,B+,4939


## Analytical Outputs

The validated summaries created during exploratory analysis are exported as CSV files.

These outputs provide a reproducible analytical layer between the raw source data and the final Excel and Power BI dashboards.

In [32]:
# Export validated analytical outputs

monthly_appointments.to_csv(
    cleaned_path / "monthly_appointments.csv",
    index=False
)

monthly_status.to_csv(
    cleaned_path / "monthly_status.csv",
    index=False
)

clinic_activity.to_csv(
    cleaned_path / "clinic_activity.csv",
    index=False
)

doctor_activity.to_csv(
    cleaned_path / "doctor_activity.csv",
    index=False
)

specialty_activity.to_csv(
    cleaned_path / "specialty_activity.csv",
    index=False
)

kpi_summary.to_csv(
    cleaned_path / "kpi_summary.csv",
    index=False
)

treatment_summary.to_csv(
    cleaned_path / "treatment_summary.csv",
    index=False
)

treatment_status_summary.to_csv(
    cleaned_path / "treatment_status_summary.csv",
    index=False
)

payment_summary.to_csv(
    cleaned_path / "payment_summary.csv",
    index=False
)

financial_summary.to_csv(
    cleaned_path / "financial_summary.csv",
    index=False
)

insurance_summary.to_csv(
    cleaned_path / "insurance_summary.csv",
    index=False
)

insurance_provider_summary.to_csv(
    cleaned_path / "insurance_provider_summary.csv",
    index=False
)

coverage_summary.to_csv(
    cleaned_path / "coverage_summary.csv",
    index=False
)

gender_summary.to_csv(
    cleaned_path / "gender_summary.csv",
    index=False
)

blood_group_summary.to_csv(
    cleaned_path / "blood_group_summary.csv",
    index=False
)

print("Analytical outputs exported successfully.")

Analytical outputs exported successfully.


## Key Analytical Definitions

| Measure | Definition |
|---|---|
| Total appointments | Count of unique appointment IDs |
| Completed appointments | Appointments with status = Completed |
| Cancelled appointments | Appointments with status = Cancelled |
| Pending appointments | Appointments with status = Pending |
| Completion rate | Completed appointments / total appointments |
| Cancellation rate | Cancelled appointments / total appointments |
| Unique patients | Distinct patients appearing in the appointment table |
| Total treatment cost | Sum of recorded treatment costs |
| Average treatment cost | Mean recorded treatment cost |
| Recorded payment value | Sum of payment amounts |
| Average payment | Mean payment amount |

The dataset does not contain a `No-show` category, so no-show rate is not calculated.

Payment amount is described as recorded payment value rather than accounting revenue because payment dates can occur before the associated appointment date.